# Step 2b — Real cross-source address pairs (Overture + Foursquare)

Builds positive address pairs for CT: the same place's address as published by Overture (side A, usually Meta/Microsoft text) vs. Foursquare's own record (side B).

**Pipeline**
1. `ct_places` — Overture places in CT (release `2026-09-23.1`)
2. `ct_bridge` — GERS id → Foursquare / AllThePlaces record ids (same release)
3. `fsq_ct` — Foursquare OS Places records around CT (release `dt=2026-09-15`)
4. `ct_positives` — the 3-way join → positive pairs
5. Analysis of the positives
6. `ct_positives_clean` — address cleaning (units, floors, punctuation) + `same_street_address` flag

⚠️ Cells marked **RUN ONCE** download from S3 / Hugging Face and take minutes. All outputs are saved to `Overture/`; later cells read the local parquet files.

In [2]:
import os
import duckdb

con = duckdb.connect()                     # in-memory; tables vanish when the kernel restarts
con.sql("INSTALL httpfs; LOAD httpfs;")    # lets DuckDB read s3:// and https:// paths
con.sql("SET s3_region = 'us-west-2';")    # Overture's public bucket region

## 1. Overture places in CT — `ct_places`

- Bounding box around CT for speed (lets DuckDB skip row groups), then an exact CT filter.
- `region` values were `CT` and `ct` → compare with `UPPER()`.
- 393 places with NULL `region` but a CT ZIP (`06…`) are kept; the rest of the NULL-region rows are NY/MA/RI or have no postcode.
- `confidence` (does the place exist?) is kept as a column, not filtered — it says nothing about address quality.

Result: **193,270 places**, all ids unique.

In [ ]:
# RUN ONCE — scans Overture places on S3 (~11 GB total, but only CT row groups/columns are fetched)
con.sql("""
CREATE TABLE ct_places AS
SELECT id, names.primary AS name, UPPER(addresses[1].freeform) AS freeform, UPPER(addresses[1].locality) AS locality, addresses[1].postcode AS postcode, addresses[1].region AS region, confidence, sources, bbox
FROM read_parquet('s3://overturemaps-us-west-2/release/2026-09-23.1/theme=places/type=place/*.zstd.parquet')
WHERE addresses[1].freeform IS NOT NULL
AND
    bbox.xmin BETWEEN -73.73 AND -71.78 AND
    bbox.ymin BETWEEN 40.98 AND 42.06 AND
    -- parentheses matter: AND binds tighter than OR
    (UPPER(addresses[1].region) = 'CT'
    OR (addresses[1].region IS NULL AND addresses[1].postcode LIKE '06%'))
""")

con.sql("""
    COPY ct_places TO
    'Overture/ct_places_2026-09-23.1.parquet' (FORMAT parquet)
""")

In [ ]:
# Check: expect 193,270
con.sql("""
    SELECT COUNT(*) FROM 
    read_parquet('Overture/ct_places_2026-09-23.1.parquet')
""")

## 2. Bridge links — `ct_bridge`

Overture's published record keeps only the provider that "won" the merge (`sources[1]`, verified `property = ''` for every row). The **bridge files** list every provider record that was merged into each GERS id — that's what turns single records into pairs.

- `provider` = whose record this bridge row points to (side B candidate; comes from the `provider=…` folder name)
- `published_from` = whose address Overture published (side A)
- `provider = published_from` → **self-pair** (same text on both sides) → not usable

Result: 58,694 rows (50,181 Foursquare + 8,513 AllThePlaces, one record per provider per place). Self-pairs: 12,483 FSQ + 2,067 ATP → **44,144 usable**.

In [ ]:
# RUN ONCE — scans the FSQ (~800 MB) and ATP (~210 MB) bridge files on S3
# UNION ALL stacks the two results; columns are matched by POSITION, so both SELECTs must list them in the same order
con.sql("""
    CREATE TABLE ct_bridge AS 

    SELECT p.id AS gers_id, b.record_id, b.provider, LOWER(p.sources[1].dataset) AS published_from
    FROM read_parquet('Overture/ct_places_2026-09-23.1.parquet') AS p
    JOIN read_parquet('s3://overturemaps-us-west-2/bridgefiles/2026-09-23.1/provider=foursquare/theme=places/type=place/*.parquet') AS b
    ON p.id = b.id

    UNION ALL

    SELECT p.id AS gers_id, b.record_id, b.provider, LOWER(p.sources[1].dataset) AS published_from
    FROM read_parquet('Overture/ct_places_2026-09-23.1.parquet') AS p
    JOIN read_parquet('s3://overturemaps-us-west-2/bridgefiles/2026-09-23.1/provider=alltheplaces/theme=places/type=place/*.parquet') AS b
    ON p.id = b.id;

    COPY ct_bridge TO 'Overture/ct_bridge_2026-09-23.1.parquet' (FORMAT parquet)
""")

In [ ]:
# Check: who supplied side A vs. which provider record we link to; rows where the two match are self-pairs
con.sql("""
SELECT provider, published_from, COUNT(*) as N
FROM read_parquet('Overture/ct_bridge_2026-09-23.1.parquet')
GROUP BY provider, published_from
""")

## 3. Foursquare records around CT — `fsq_ct`

Foursquare OS Places is gated on Hugging Face: the token comes from `HF_TOKEN` (set in the git-ignored `.env`, loaded by direnv). Never print it.

- **Bounding box only, no region filter.** Some Foursquare places located in CT carry wrong states (`KS`, `CA`, …); a region filter would drop them and lose their pairs. CT-ness was already decided on the Overture side — this table is just a lookup, and the join does the exact selection.
- `region` is kept raw (`CT` / `Connecticut` / `ct` …) — normalize it in feature code, not here.

Coverage check: only **31 of 50,181** bridge Foursquare ids are missing from `fsq_ct` (99.94%) — the id formats match.

In [ ]:
# Register the Hugging Face token with DuckDB (temporary: lasts for this connection only)
con.sql(f"CREATE SECRET hf (TYPE huggingface, TOKEN '{os.environ['HF_TOKEN']}')")

In [ ]:
# RUN ONCE — reads Foursquare OS Places from Hugging Face
con.sql("""
    CREATE TABLE fsq_ct AS 
    SELECT 
    fsq_place_id, name,
    UPPER(address) AS address, 
    UPPER(locality) AS locality,
    region, postcode,
    latitude, longitude, bbox,
    date_closed
    FROM 
    read_parquet('hf://datasets/foursquare/fsq-os-places/release/dt=2026-09-15/places/parquet/*.parquet')
    WHERE 
    bbox.xmin BETWEEN -73.73 AND -71.78 AND
    bbox.ymin BETWEEN 40.98 AND 42.06 
"""
)

con.sql("""
    COPY fsq_ct
    TO 'Overture/fsq_ct_2026-09-15.parquet' (FORMAT PARQUET)
""")

In [ ]:
# Coverage check: LEFT JOIN keeps every bridge row, so misses show up as NULLs (expect 31 True)
con.sql("""
    SELECT(q.fsq_place_id IS NULL) as fsq_id_null, COUNT(*) AS N
    FROM (
        SELECT *
        FROM read_parquet('Overture/ct_bridge_2026-09-23.1.parquet') 
        where provider = 'foursquare'
    ) as p
    LEFT JOIN read_parquet('Overture/fsq_ct_2026-09-15.parquet') as q
    ON p.record_id = q.fsq_place_id
    GROUP BY fsq_id_null
""").df()

## 4. Positive pairs — `ct_positives`

Chain of joins through the bridge (the middle table): `places ── gers_id ── bridge ── record_id ── foursquare`.
Foursquare only for now (AllThePlaces deferred — see open questions). Self-pairs excluded.

Result: **37,680 pairs**. `identical_address` flags the 70% (26,286) whose strings are exactly equal — kept, but report results on the non-identical subset.

In [ ]:
con.sql("""
    DROP TABLE IF EXISTS ct_positives;
    CREATE TABLE ct_positives AS 
    SELECT 
    b.gers_id, f.fsq_place_id,            -- ids kept for tracing pairs in error analysis
    p.freeform AS address_a,              -- _a = Overture (published_from provider)
    f.address AS address_b,               -- _b = Foursquare
    p.locality AS locality_a,
    f.locality AS locality_b,
    p.region AS region_a,
    f.region AS region_b,
    p.postcode AS postcode_a,
    f.postcode AS postcode_b,
    p.name AS name_a,                     -- names are NOT features (address-only scope); for reading pairs only
    f.name AS name_b,
    b.published_from,
    1 AS label,
    address_a = address_b AS identical_address
    FROM read_parquet('Overture/ct_bridge_2026-09-23.1.parquet') AS b
    JOIN read_parquet('Overture/ct_places_2026-09-23.1.parquet') AS p 
    ON b.gers_id = p.id
    JOIN read_parquet('Overture/fsq_ct_2026-09-15.parquet') AS f
    on b.record_id = f.fsq_place_id
    WHERE b.provider = 'foursquare' AND b.provider <> b.published_from;

    COPY ct_positives TO 'Overture/ct_positives_2026-09-23.1.parquet'
    (FORMAT parquet);
""")

## 5. What's in the positives

Eyeball sample of the non-identical pairs. Categories seen: unit/suite added on one side, abbreviations, ranges (`16-18`), ZIP+4, junk (`\N\N`), real typos (`DANBURY EOAD`), locality variants (`SPRAGUE`/`BALTIC`), and different house numbers.

In [ ]:
# Random sample of pairs that actually differ (re-run for a new sample; SELECT setseed(x) first to make it repeatable)
con.sql("""
    SELECT address_a, address_b, locality_a, locality_b, postcode_a, postcode_b, name_a, name_b
    FROM read_parquet('Overture/ct_positives_2026-09-23.1.parquet')
    WHERE NOT identical_address
    ORDER BY random()
    LIMIT 20
""").df()

House-number comparison. Result: 1,177 missing a number on one side (ambiguous — keep, flag) and **1,778 with different numbers** (likely label noise — flag, keep out of training positives, sample for the hand-labeled test set). The 544 rows with NULL `address_b` fall in neither count (NULL fails every comparison).

In [74]:
# Leading house number on each side; regexp_extract returns '' (not NULL) when there's no match
con.sql(r"""
SELECT 
    COUNT(*) 
        AS total,
    COUNT(*) FILTER (WHERE number_a = '' OR number_b = '') 
        AS missing_number,
    COUNT(*) FILTER 
        (WHERE number_a <> '' AND number_b <> '' AND number_a <> number_b)                 
        AS different_number
FROM  
    ( 
    SELECT 
        regexp_extract(address_a, '^\d+') AS number_a,
        regexp_extract(address_b, '^\d+') AS number_b
    FROM 
        read_parquet('Overture/ct_positives_2026-09-23.1.parquet')
)
"""
)

┌───────┬────────────────┬──────────────────┐
│ total │ missing_number │ different_number │
│ int64 │     int64      │      int64       │
├───────┼────────────────┼──────────────────┤
│ 37680 │           1177 │             1778 │
└───────┴────────────────┴──────────────────┘

## 6. Address cleaning — `ct_positives_clean`

Needed for the **building-level** match definition: same street address = same location, units ignored.

**Decisions**
- Floors are stripped (inside the building, like a suite).
- `BLDG` / `BUILDING` is **kept** — a different building number counts as a different location.
- Road-number words are **not** designators: `ROUTE`/`RTE`/`RT`/`US`/`SR`/`CT`/`CONNECTICUT`/`I` (`ROUTE 1`, `I 95`). `BOX` (PO box) not stripped either.
- Punctuation is replaced with a **space**, not deleted (`16-18`, `RD,SUITE` must not glue together). `-` is kept.

**Designator list** — found by counting the word before a digit-containing last token (`STE` 1,656, `UNIT` 262, `SUITE` 143, `FL` 82, `SPC` 36, …), filtered by hand:
`SUITE STE UITE UNIT APARTMENT APT FLOOR FL SPACE SPC ROOM RM OFFICE OFC STORE PMB NUMBER PAD BAY`

The two cells below are the exploration behind two of the decisions; 6c onward is the pipeline.

### 6a. Units with no designator and no `#`

Does `… RD 2` / `… ST D16` happen often enough to need a rule? Last tokens after `RD`/`ST` are mostly directionals (`… MAIN ST N`), which must **not** be stripped. Only 52 bare unit codes → known gap.

In [36]:
# 6a. What comes after a street type? second = 2nd-to-last word, last = last word
# Result: mostly directionals (RD 317, ST 184) — only 52 bare unit codes (~0.1% of rows) → known gap, no rule
con.sql(r"""
    WITH t AS (
    SELECT 
        regexp_extract(address_a, '.*\s([A-Za-z0-9]+)\s[A-Za-z0-9]+$', 1) AS second, 
        regexp_extract(address_a, '.*\s([A-Za-z0-9]+)$', 1) AS last,  
    FROM read_parquet('Overture/ct_positives_2026-09-23.1.parquet') 
    )
    SELECT
        second, 
        CASE 
            WHEN UPPER(last) IN ('N', 'S', 'E', 'W', 'NE', 'NW', 'SE', 'SW', 'NORTH', 'SOUTH', 'EAST', 'WEST', 'NORTHEAST', 'NORTHWEST', 'SOUTHEAST', 'SOUTHWEST') THEN 'directional'
            WHEN regexp_matches(last, '\d') THEN 'has_digit'
            ELSE 'other'
        END AS kind,
        COUNT(*) AS n
    FROM t
    WHERE second IN ('RD', 'ST')
    GROUP BY ALL
    ORDER BY second, n DESC
""").df()

,second,KIND,n
0,RD,directional,317
1,RD,has_digit,34
2,RD,other,13
3,ST,directional,184
4,ST,other,36
5,ST,has_digit,18


### 6b. Two-word designators

`SPACE NUMBER 1029A`, `SUITE STE 3`: the single-designator pattern left the first word behind. 11 rows → the step-2 pattern allows an optional first designator.

In [56]:
# 6b. Pairs of consecutive designator words in the 3rd- and 2nd-to-last positions (e.g. SPACE NUMBER 1029A)
# Result: 11 rows → handled by an optional first designator in the step-2 pattern
con.sql(r"""
    SELECT 
       regexp_extract(address_a, '.*\s([A-Za-z0-9]+)\s([A-Za-z0-9]+)\s[A-Za-z0-9]+$', 1) AS third,  
        regexp_extract(address_a, '.*\s([A-Za-z0-9]+)\s[A-Za-z0-9]+$', 1) AS second, 
        COUNT(*) AS N
        /*regexp_extract(address_a, '.*\s([A-Za-z0-9]+)$', 1) AS last*/
    FROM read_parquet('Overture/ct_positives_2026-09-23.1.parquet') 
    WHERE 
    third IN ('SUITE', 'STE', 'UITE', 'UNIT', 'APARTMENT', 'APT', 'FLOOR', 'FL', 'SPACE', 'SPC', 'ROOM', 'RM', 'OFFICE', 'OFC', 'STORE', 'PMB', 'NUMBER', 'PAD', 'BAY') AND
    second IN ('SUITE', 'STE', 'UITE', 'UNIT', 'APARTMENT', 'APT', 'FLOOR', 'FL', 'SPACE', 'SPC', 'ROOM', 'RM', 'OFFICE', 'OFC', 'STORE', 'PMB', 'NUMBER', 'PAD', 'BAY')
    GROUP BY third, second
    ORDER BY N DESC
""").df()

,third,second,N
0,SUITE,STE,3
1,UNIT,STE,2
2,FLOOR,SUITE,1
3,STORE,STE,1
4,STE,UNIT,1
5,FLOOR,UITE,1
6,SPACE,STE,1
7,SPACE,NUMBER,1


### 6c. Cleaning chain + gap check

Order: floor → designator → `#` → punctuation → whitespace. Unit stripping must run **before** punctuation removal (it needs `#` and `-`).

Gap check = search the cleaned output for anything that should be gone. Remaining (both sides, out of 37,680), all accepted as **known gaps**:

| Leftover | A | B | Typical shape |
|---|---|---|---|
| designator word | 22 | 77 | two units (`SUITE 240 STE 700R`), code with a space (`SUITE G 110`), unit before `BLDG`, false alarms (`POST OFFICE SQ`) |
| city/state/ZIP in address | 44 | 101 | `… BRIDGEPORT CT 06607` |
| floor | 8 | 3 | `GROUND FLOOR`, `UPPER FLOOR`, `2ND STE 7 FLOOR` |
| `#` mid-address (before step 4) | 21 | 21 | `UNIT #1 & #2` |

Also known: 52 bare unit codes (6a), a raw newline inside one address (`233 ELM STREET\n2ND FLOOR`), free-text descriptions instead of addresses.

In [96]:
# 6c. Cleaning chain + gap check — each step reads the previous step's column, so a broken row shows which step failed
con.sql(r"""
WITH
    -- 1. ordinal floors at the end: 2ND FLOOR, 14TH FL (runs first so STE 5 2ND FLOOR -> STE 5 -> stripped in step 2)
    step1 AS (
            SELECT 
            *,
            regexp_replace(address_a, '\s+[A-Za-z0-9]+\s+(FLOOR|FL)$', '') AS a_remove_fl,
            regexp_replace(address_b, '\s+[A-Za-z0-9]+\s+(FLOOR|FL)$', '') AS b_remove_fl
            FROM read_parquet('Overture/ct_positives_2026-09-23.1.parquet')
        ),
    -- 2. designator + unit code at the end, optionally two designators: STE 200, SUITE #320, SPACE NUMBER 1029A
    step2 AS (
        SELECT
            *,
            regexp_replace(a_remove_fl, 
                '\s+((SUITE|STE|UITE|UNIT|APARTMENT|APT|FLOOR|FL|SPACE|SPC|ROOM|RM|OFFICE|OFC|STORE|PMB|NUMBER|PAD|BAY)\s)?(SUITE|STE|UITE|UNIT|APARTMENT|APT|FLOOR|FL|SPACE|SPC|ROOM|RM|OFFICE|OFC|STORE|PMB|NUMBER|PAD|BAY)\s[#a-zA-Z0-9-]+$',
                ''
            ) AS a_remove_des,
            regexp_replace(b_remove_fl, 
                '\s+((SUITE|STE|UITE|UNIT|APARTMENT|APT|FLOOR|FL|SPACE|SPC|ROOM|RM|OFFICE|OFC|STORE|PMB|NUMBER|PAD|BAY)\s)?(SUITE|STE|UITE|UNIT|APARTMENT|APT|FLOOR|FL|SPACE|SPC|ROOM|RM|OFFICE|OFC|STORE|PMB|NUMBER|PAD|BAY)\s[#a-zA-Z0-9-]+$',
                ''
            ) AS b_remove_des
        FROM step1
    ),
    -- 3. bare # units at the end: #204, # 1D
    step3 AS (
        SELECT
        *,
        regexp_replace(a_remove_des, '\s*\#\s*[A-Z0-9]+$', '') AS a_remove_hash,
        regexp_replace(b_remove_des, '\s*\#\s*[A-Z0-9]+$', '') AS b_remove_hash,
        FROM step2
    ),
    -- 4. punctuation -> space ('g' = replace every match); keeps letters, digits, '-', spaces
    step4 AS (
        SELECT 
        *,
        regexp_replace(a_remove_hash, '[^A-Z0-9- ]', ' ', 'g') AS a_remove_punctuation,
        regexp_replace(b_remove_hash, '[^A-Z0-9- ]', ' ', 'g') AS b_remove_punctuation,
        FROM step3
    ),
    -- 5. collapse whitespace runs, trim ends
    step5 AS (
        SELECT
        *, 
        TRIM(regexp_replace(a_remove_punctuation, '\s+', ' ', 'g')) AS a_remove_spaces,
        TRIM(regexp_replace(b_remove_punctuation, '\s+', ' ', 'g')) AS b_remove_spaces,
        FROM step4
    )
-- Gap check: what survived cleaning (side B; swap b_ -> a_ for side A)
SELECT 
    COUNT(*) FILTER (WHERE 
        regexp_matches(b_remove_spaces, '\s+$')) AS trailing_space,
    COUNT(*) FILTER (WHERE 
        regexp_matches(b_remove_spaces, '^\s+')) AS leading_spacing,
    COUNT(*) FILTER (WHERE 
        regexp_matches(b_remove_spaces, '\s(SUITE|STE|UITE|UNIT|APARTMENT|APT|FLOOR|FL|SPACE|SPC|ROOM|RM|OFFICE|OFC|STORE|PMB|NUMBER|PAD|BAY)(\s|$)')) AS designator_end,
    COUNT(*) FILTER (WHERE 
        regexp_matches(b_remove_hash, '\#')) AS hash_exists_before_punc,
    COUNT(*) FILTER (WHERE 
        regexp_matches(b_remove_spaces, '\#')) AS hash_exists,
    COUNT(*) FILTER (WHERE
        regexp_matches(b_remove_spaces, 'CT\s+[\d-]{5,}')) AS trailing_zip,
    COUNT(*) FILTER (WHERE
        regexp_matches(b_remove_spaces, '\s(FL|FLOOR)$')) AS trailing_floor,
FROM step5
"""
).df()

,trailing_space,leading_spacing,designator_end,hash_exists_before_punc,hash_exists,trailing_zip,trailing_floor
0,0,0,77,21,0,101,3


### 6d. Save — `ct_positives_clean_2026-09-23.1.parquet`

18 columns: the 15 from `ct_positives` + `address_a_clean`, `address_b_clean`, `same_street_address` (cleaned strings equal; NULL when `address_b` is NULL).

In [98]:
# 6d. RUN ONCE — same chain as 6c, written out with all 15 original columns + the cleaned addresses + flag
# Keep the cleaning steps identical to 6c if either cell changes
con.sql(r"""
COPY
(WITH
    step1 AS (
            SELECT 
            *,
            regexp_replace(address_a, '\s+[A-Za-z0-9]+\s+(FLOOR|FL)$', '') AS a_remove_fl,
            regexp_replace(address_b, '\s+[A-Za-z0-9]+\s+(FLOOR|FL)$', '') AS b_remove_fl
            FROM read_parquet('Overture/ct_positives_2026-09-23.1.parquet')
        ),
    step2 AS (
        SELECT
            *,
            regexp_replace(a_remove_fl, 
                '\s+((SUITE|STE|UITE|UNIT|APARTMENT|APT|FLOOR|FL|SPACE|SPC|ROOM|RM|OFFICE|OFC|STORE|PMB|NUMBER|PAD|BAY)\s)?(SUITE|STE|UITE|UNIT|APARTMENT|APT|FLOOR|FL|SPACE|SPC|ROOM|RM|OFFICE|OFC|STORE|PMB|NUMBER|PAD|BAY)\s[#a-zA-Z0-9-]+$',
                ''
            ) AS a_remove_des,
            regexp_replace(b_remove_fl, 
                '\s+((SUITE|STE|UITE|UNIT|APARTMENT|APT|FLOOR|FL|SPACE|SPC|ROOM|RM|OFFICE|OFC|STORE|PMB|NUMBER|PAD|BAY)\s)?(SUITE|STE|UITE|UNIT|APARTMENT|APT|FLOOR|FL|SPACE|SPC|ROOM|RM|OFFICE|OFC|STORE|PMB|NUMBER|PAD|BAY)\s[#a-zA-Z0-9-]+$',
                ''
            ) AS b_remove_des
        FROM step1
    ),
    step3 AS (
        SELECT
        *,
        regexp_replace(a_remove_des, '\s*\#\s*[A-Z0-9]+$', '') AS a_remove_hash,
        regexp_replace(b_remove_des, '\s*\#\s*[A-Z0-9]+$', '') AS b_remove_hash,
        FROM step2
    ),
    step4 AS (
        SELECT 
        *,
        regexp_replace(a_remove_hash, '[^A-Z0-9- ]', ' ', 'g') AS a_remove_punctuation,
        regexp_replace(b_remove_hash, '[^A-Z0-9- ]', ' ', 'g') AS b_remove_punctuation,
        FROM step3
    ),
    step5 AS (
        SELECT
        *, 
        TRIM(regexp_replace(a_remove_punctuation, '\s+', ' ', 'g')) AS a_remove_spaces,
        TRIM(regexp_replace(b_remove_punctuation, '\s+', ' ', 'g')) AS b_remove_spaces,
        FROM step4
    )
SELECT
    gers_id,
    fsq_place_id,
    address_a,
    address_b,
    locality_a,
    locality_b,
    region_a,
    region_b,
    postcode_a,
    postcode_b,
    name_a,
    name_b,
    published_from,
    label, 
    identical_address,
    a_remove_spaces AS address_a_clean,
    b_remove_spaces AS address_b_clean,
    address_a_clean = address_b_clean AS same_street_address,
FROM step5
) TO 'Overture/ct_positives_clean_2026-09-23.1.parquet' (FORMAT parquet)
"""
)

### 6e. Checks

Result: 37,680 rows, 544 NULL. **TF = 0** (nothing broke), TT = 26,286, **FT = 3,516**.

- Same street address: **29,802** (79%) vs. 26,286 (70%) identical raw strings.
- Of the 10,850 non-identical pairs with both addresses present, **32%** differed only by formatting/units.
- **7,334** positives still differ after cleaning — abbreviations, typos, different house numbers, city/ZIP in the address. These are the hard positives for the model.

In [ ]:
# Expect 18 columns
con.sql("""
DESCRIBE SELECT *
FROM read_parquet('Overture/ct_positives_clean_2026-09-23.1.parquet')
""")

In [ ]:
# Expect 37,680 total; 544 NULL on both flags (NULL address_b)
con.sql("""
SELECT
    COUNT(*) AS total,
    COUNT(*) FILTER (identical_address = TRUE) AS identical_addresses,
    COUNT(*) FILTER (identical_address IS NULL) AS identical_addresses_null,
    COUNT(*) FILTER (same_street_address = TRUE) AS same_street_address,
    COUNT(*) FILTER (same_street_address IS NULL) AS same_street_address_null
FROM read_parquet('Overture/ct_positives_clean_2026-09-23.1.parquet')
""")

In [ ]:
# Sanity check: cleaning applies the same steps to both sides, so identical-before must stay identical-after (TF must be 0)
# FT = pairs that only differed by units / floors / # / punctuation
con.sql("""
SELECT
    COUNT(*) FILTER (identical_address AND NOT same_street_address)
    AS TF,
    COUNT(*) FILTER (identical_address AND same_street_address)
    AS TT,
    COUNT(*) FILTER (NOT identical_address AND same_street_address)
    AS FT,
FROM read_parquet('Overture/ct_positives_clean_2026-09-23.1.parquet')
""")

┌───────┬───────┬───────┐
│  TF   │  TT   │  FT   │
│ int64 │ int64 │ int64 │
├───────┼───────┼───────┤
│     0 │ 26286 │  3516 │
└───────┴───────┴───────┘